In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

# Load the processed dataset
df = pd.read_csv('../data/processed/RuralCreditData_processed.csv')

print("Processed dataset loaded. Ready for Statistical Analysis.")

In [ ]:
# Select only numerical columns for correlation
num_df = df.select_dtypes(include=[np.number])
corr_matrix = num_df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu', fmt=".2f", linewidths=0.5)
plt.title('Financial Correlation Heatmap')
plt.show()

**Analysis:** I am analyzing the correlation between `annual_income`, `disposable_income`, and `loan_amount`. A high positive correlation between income and loan amount would suggest a traditional lending model, while a low correlation suggests that other factors (like social class or business type) are driving loan decisions.

In [ ]:
# Let's compare two social classes: 'OBC' and 'Mochi'
group1 = df[df['social_class'] == 'OBC']['loan_amount']
group2 = df[df['social_class'] == 'Mochi']['loan_amount']

# Perform an Independent T-Test
t_stat, p_val = stats.ttest_ind(group1, group2)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

if p_val < 0.05:
    print("Result: Statistically Significant. There is a real difference in loan amounts between these groups.")
else:
    print("Result: Not Statistically Significant. Any difference is likely due to chance.")

**Statistical Insight:** I performed an Independent T-Test to determine if the `social_class` significantly impacts the `loan_amount`. With a p-value of [Insert your p-value], we can conclude whether the institution is biased toward a specific social class or if lending is equitable.

In [ ]:
# Define X (Income) and y (Loan Amount)
X = df['annual_income']
y = df['loan_amount']

# Add a constant for the intercept
X = sm.add_constant(X)

# Fit the regression model
model = sm.OLS(y, X).fit()

print(model.summary())

# Plot the regression line
plt.figure(figsize=(10, 6))
sns.regplot(x='annual_income', y='loan_amount', data=df, line_kws={"color": "red"})
plt.title('Linear Regression: Annual Income vs Loan Amount')
plt.show()

**Analysis:** The regression model shows the relationship between income and loan size. The R-squared value tells us how much of the loan amount is explained by income. A low R-squared indicates that the bank uses non-traditional factors (like assets or business type) to decide loans, which is typical for rural lending.

In [ ]:
# Define a simple risk scoring formula
# Risk increases if: Low disposable income, High debt-to-income ratio, High dependents
df['risk_score'] = (df['loan_to_income_ratio'] * 0.5) + (df['total_dependents'] * 10) - (df['disposable_income'] / 1000)

# Categorize risk
def categorize_risk(score):
    if score > 50: return 'High Risk'
    elif score > 20: return 'Medium Risk'
    else: return 'Low Risk'

df['risk_category'] = df['risk_score'].apply(categorize_risk)

# Show the distribution of risk
print(df['risk_category'].value_counts())

**Business Recommendation:** I have developed a custom Risk Scoring Model. By combining disposable income and dependents, we can now flag "High Risk" borrowers before the loan is disbursed, potentially reducing default rates by X%.